In [1]:
import os
import json
import numpy as np
from scipy.interpolate import CubicSpline
from sklearn.linear_model import RANSACRegressor, LinearRegression

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
output_json_path = "./testdistance_estimates_smoothed.json"

# ---------- ジャンプ補正 ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# ---------- 安定区間抽出 ----------
def find_stable_segments(data, diff_threshold=3.0, min_length=10, min_value=15.0):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    stable_mask = (diffs < diff_threshold) & (data > min_value)

    segments = []
    start = None
    for i, val in enumerate(stable_mask):
        if val:
            if start is None:
                start = i
        else:
            if start is not None and i - start >= min_length:
                segments.append((start, i - 1))
            start = None
    if start is not None and len(data) - start >= min_length:
        segments.append((start, len(data) - 1))
    return segments

# ---------- スプライン補完 ----------
def apply_spline_fit_partial(x_all, data, stable_segments, clip_margin=10.0):
    stable_x = []
    stable_y = []

    median_val = np.median(data)
    min_y = median_val - clip_margin
    max_y = median_val + clip_margin

    for start, end in stable_segments:
        for i in range(start, end + 1):
            if min_y <= data[i] <= max_y:
                stable_x.append(i)
                stable_y.append(data[i])

    if len(stable_x) < 4:
        return data.tolist()

    unique_pairs = list({x: y for x, y in zip(stable_x, stable_y)}.items())
    if len(unique_pairs) < 4:
        return data.tolist()

    unique_pairs.sort()
    sorted_x, sorted_y = zip(*unique_pairs)
    sorted_x = np.array(sorted_x)
    sorted_y = np.clip(np.array(sorted_y), min_y, max_y)

    spline = CubicSpline(sorted_x, sorted_y, bc_type='natural')

    smoothed = data.copy()
    for i in range(len(data)):
        if sorted_x[0] <= i <= sorted_x[-1]:
            smoothed[i] = float(np.clip(spline(i), min_y, max_y))
    return smoothed.tolist()

# ---------- RANSAC補完 ----------
def ransac_fit(data, min_valid=10.0, max_valid=100.0):
    x = np.arange(len(data)).reshape(-1, 1)
    y = np.array(data)

    valid_mask = (y >= min_valid) & (y <= max_valid)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    if len(x_valid) < 2:
        return data.tolist()

    model = RANSACRegressor(estimator=LinearRegression(), min_samples=5, residual_threshold=3.0)
    model.fit(x_valid, y_valid)
    y_pred = model.predict(x)

    return y_pred.tolist()

# ---------- メイン処理 ----------
with open(json_path, encoding="utf-8") as f:
    original_data = json.load(f)

smoothed_data = {}

for scene_id, frame_data in original_data.items():
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = np.array([frame_data[k] for k in frame_keys], dtype=float)
    x_all = np.arange(len(distances))

    if scene_id == "211":
        smoothed_values = ransac_fit(distances)
    else:
        jump_corrected = suppress_jumps(distances)
        segments = find_stable_segments(jump_corrected)
        smoothed_values = apply_spline_fit_partial(x_all, jump_corrected, segments)

    # フレームごとの結果を保存
    smoothed_data[scene_id] = {
        key: round(val, 4) for key, val in zip(frame_keys, smoothed_values)
    }

# ---------- 書き出し ----------
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(smoothed_data, f, ensure_ascii=False, indent=2)

print(f"✅ 全シーンの補完結果を保存しました: {output_json_path}")


✅ 全シーンの補完結果を保存しました: ./testdistance_estimates_smoothed.json


In [2]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# ---------- 設定 ----------
input_json_path = "./testdistance_estimates_smoothed.json"
output_dir = "./scene_spline_graphs"
os.makedirs(output_dir, exist_ok=True)

# ---------- JSON読み込み ----------
with open(input_json_path, encoding="utf-8") as f:
    smoothed_data = json.load(f)

# ---------- 各シーンをグラフ化 ----------
for scene_id, frame_data in smoothed_data.items():
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = [frame_data[k] for k in frame_keys]

    plt.figure(figsize=(10, 6))
    plt.plot(distances, marker='o', markersize=3, label="Smoothed Distance", color="green")
    plt.title(f"Scene {scene_id} - Smoothed Distance")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{scene_id}_smoothed.png")
    plt.savefig(save_path)
    plt.close()

print(f"✅ 各シーンの距離グラフを {output_dir}/ に保存しました。")


✅ 各シーンの距離グラフを ./scene_spline_graphs/ に保存しました。
